In [0]:
%sql
-- Setting up local mirror of workspace.superstore schema
CREATE SCHEMA IF NOT EXISTS workspace.superstore;

In [0]:
%sql
-- Copying online raw data into expected bronze table path
CREATE TABLE IF NOT EXISTS workspace.superstore.bronze_online_superstore
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
AS
SELECT * FROM dbacademy.Superstore.sales_online_raw;

In [0]:
# Setting correct data types, fixing Product ID uniqueness, and deduplicating records

df_online_bronze = spark.read.table("workspace.superstore.bronze_online_superstore")
df_online_bronze.createOrReplaceTempView("bronze_superstore_view")

df_online_silver = (spark.sql("""
    WITH RankedProducts AS (
        SELECT *,
               DENSE_RANK() OVER (PARTITION BY TRIM(`Product ID`) ORDER BY TRIM(`Product Name`)) as name_rank
        FROM bronze_superstore_view
        WHERE `Order ID` is not null
    )
    SELECT 
        `Row ID` AS row_id,
        `Order ID` AS receipt_id,
        to_timestamp(`Order Date`, 'M/d/yyyy') AS transaction_timestamp,
        to_timestamp(`Ship Date`, 'M/d/yyyy') AS ship_date,
        `Ship Mode` AS ship_mode,
        TRIM(`Customer ID`) AS customer_id,
        `Customer Name` AS customer_name,
        `Segment` AS segment,
        `Country` AS country,
        `City` AS city,
        `State` AS state,
        `Postal Code` AS postal_code,
        `Region` AS region,
        CASE 
            WHEN name_rank = 1 THEN TRIM(`Product ID`)
            ELSE concat(
                substring_index(TRIM(`Product ID`), '-', 2),
                '-', 
                substring_index(TRIM(`Product ID`), '-', -1),
                name_rank                                     
            ) 
        END AS product_id,
        `Category` AS category,
        `Sub-Category` AS sub_category,
        `Product Name` AS product_name,
        `Sales` AS sales_price,
        `Quantity` AS quantity,
        `Discount` AS discount,
        `Profit` AS profit
    FROM RankedProducts
"""))

df_online_silver = df_online_silver.dropDuplicates()



In [0]:
#Writing to silver layer
(df_online_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.superstore.silver_online_superstore") 
)